# 📈 Stock Price Forecasting — End-to-End Data Science Project

> **Goal:** Predict future stock closing prices using historical market data and technical indicators.

## Problem Statement

Stock price forecasting is a classic **time-series regression** problem. Given historical price and volume data for a stock, we want to predict the **closing price** for future trading days. This is not a classification problem (up/down) — we are predicting the *exact numerical value*.

### Why This Matters

Accurate price forecasting helps:
- **Traders** make informed entry/exit decisions
- **Portfolio managers** assess risk and rebalance
- **Algorithmic trading systems** generate signals
- **Retail investors** understand market trends

### Project Workflow

```
Step 1: Problem Framing   → Define what we're predicting and why
Step 2: Data Acquisition  → Fetch real stock data from Yahoo Finance
Step 3: EDA               → Explore trends, distributions, and patterns
Step 4: Preprocessing     → Clean data, handle missing values
Step 5: Feature Eng.      → Create technical indicators and lag features
Step 6: Modeling          → Train multiple models (Linear → XGBoost → ARIMA)
Step 7: Evaluation        → Compare models on held-out test data
Step 8: Forecasting       → Generate 30-day ahead predictions
```

> **⚠️ Important Disclaimer:** Stock prices are influenced by countless factors (news, earnings, macroeconomics, sentiment) that no single model can capture perfectly. Our models will learn *patterns* from historical data, but **past performance does not guarantee future results**. Treat this as a learning exercise, not financial advice.

---

## Dataset: Apple Inc. (AAPL) — 5 Years of Real Market Data

We'll use **yfinance** to pull real historical data for Apple stock (AAPL) from Yahoo Finance. This gives us:
- **Open, High, Low, Close, Adjusted Close, Volume** — daily granularity
- **A real-world, non-synthetic dataset** with genuine market patterns
- **Recent data** (2020–2025) capturing pre- and post-COVID market behavior


## 1. Environment Setup & Library Imports

Let's load all the libraries we'll need throughout this project.

In [1]:
# ── Data manipulation
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

# ── Data fetching
import yfinance as yf

# ── Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.dates as mdates

# ── Preprocessing & metrics
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Modeling
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

# ── Time series analysis
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

# ── Statistics
from scipy import stats

# ── Warnings
import warnings
warnings.filterwarnings('ignore')

print("\U0001f7e2 All libraries loaded successfully!")
print(f"   pandas v{pd.__version__}")
print(f"   numpy v{np.__version__}")
print(f"   sklearn v{__import__('sklearn').__version__}")
print(f"   yfinance v{yf.__version__}")


ModuleNotFoundError: No module named 'yfinance'

In [ ]:
pip install ipykernel

: 

## 2. Data Acquisition — Fetching Real Stock Data

We'll use **yfinance** to download Apple (AAPL) daily stock data.

**Why Apple (AAPL)?**
- Highly liquid, consistently traded stock
- Strong long-term trend with interesting volatility patterns
- Well-documented and widely studied
- Clean data with no survivorship bias issues

**Why 5 years?**
- Long enough to capture multiple market cycles (COVID crash, recovery, rate hikes)
- Short enough for the data to be relevant to current market dynamics
- Provides ~1,250 trading days — sufficient for training robust models

In [ ]:
# ── Define the stock ticker and date range ──
TICKER = "AAPL"
START_DATE = "2020-01-01"
END_DATE = "2025-01-01"
STOCK_NAME = "Apple Inc."

print(f"\U0001f4e1 Fetching data for {STOCK_NAME} ({TICKER})...")
print(f"   From: {START_DATE}  \u2192  To: {END_DATE}")
print("-" * 55)

# ── Download from Yahoo Finance ──
df_raw = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=False)

# ── Flatten MultiIndex columns if present ──
if isinstance(df_raw.columns, pd.MultiIndex):
    df_raw.columns = df_raw.columns.get_level_values(0)

print(f"\n\U0001f7e2 Downloaded {len(df_raw):,} trading days of data")
print(f"   Columns: {list(df_raw.columns)}")
print(f"   Date range: {df_raw.index.min()} \u2192 {df_raw.index.max()}")
df_raw.head(10)


## 3. Initial Data Inspection & Quality Check

Before any analysis, we must understand our data's structure, types, and quality. This step answers:
- **Shape**: How many rows and columns do we have?
- **Data types**: Are our columns stored correctly?
- **Missing values**: Are there gaps?
- **Basic stats**: What are the ranges, means, and standard deviations?

Stock market data has **weekend and holiday gaps** — the market only trades Mon-Fri and closes on public holidays. This is normal.

In [ ]:
# ── Shape ──
print(f"\U0001f4ca Dataset Shape: {df_raw.shape[0]:,} rows \u00d7 {df_raw.shape[1]} columns")
print()

# ── Data types ──
print("\U0001f4cb Column Data Types:")
print(df_raw.dtypes.to_string())
print()

# ── Missing values ──
missing = df_raw.isnull().sum()
print("\U0001f50d Missing Values:")
if missing.sum() == 0:
    print("   \U0001f7e2 No missing values found \u2014 clean dataset!")
else:
    print(missing[missing > 0].to_string())
print()

# ── Basic statistics ──
print("\U0001f4c8 Descriptive Statistics:")
df_raw.describe()


### 3.1 Date Regularity Check

Verifying our date index is a proper DatetimeIndex and checking the spacing between trading days.

In [ ]:
# ── Check date index properties ──
print(f"Index type: {type(df_raw.index).__name__}")
print(f"First date: {df_raw.index.min()}")
print(f"Last date:  {df_raw.index.max()}")
total_days = (df_raw.index.max() - df_raw.index.min()).days
print(f"Total calendar days in range: {total_days}")
print(f"Trading days present: {len(df_raw)}")
print(f"Weekends/holidays skipped: {total_days - len(df_raw) + 1}")
print()

# ── Check for unexpected gaps (> 5 consecutive missing trading days) ──
date_diff = pd.Series(df_raw.index).diff().dropna()
unexpected_gaps = date_diff[date_diff > pd.Timedelta(days=5)]
print(f"Unexpected gaps (>5 days gap): {len(unexpected_gaps)}")
if len(unexpected_gaps) > 0:
    print("   Gap dates:")
    for idx_val in unexpected_gaps.index:
        start = df_raw.index[idx_val - 1].strftime('%Y-%m-%d')
        end = df_raw.index[idx_val].strftime('%Y-%m-%d')
        gap_days = date_diff.iloc[idx_val - 1].days
        print(f"   {start} \u2192 {end} ({gap_days} days gap)")


## 4. Exploratory Data Analysis (EDA)

EDA is where we *listen to the data*. Before building any models, we need to understand:
- **Trend**: Is the stock generally going up or down over time?
- **Volatility**: How much does the price fluctuate?
- **Volume patterns**: When does trading activity spike?
- **Distributions**: Are prices normally distributed or skewed?
- **Correlations**: How do Open/High/Low/Close/Volume relate?

These insights guide feature engineering and model selection.

### 4.1 Price Trend Over Time

The **closing price** is our target variable. Let's visualize it alongside volume.

In [ ]:
# ── Create a dual-axis plot: Price + Volume ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), gridspec_kw={'height_ratios': [3, 1]})
fig.suptitle(f'{STOCK_NAME} ({TICKER}) \u2014 Price & Volume Over Time', fontsize=16, fontweight='bold')

# ── Price plot ──
ax1.plot(df_raw.index, df_raw['Close'], color='#1f77b4', linewidth=1.5, label='Close Price')
ax1.plot(df_raw.index, df_raw['Close'].rolling(50).mean(), color='orange', linewidth=1, alpha=0.8, label='50-day SMA')
ax1.plot(df_raw.index, df_raw['Close'].rolling(200).mean(), color='red', linewidth=1, alpha=0.8, label='200-day SMA')
ax1.fill_between(df_raw.index, df_raw['Low'], df_raw['High'], alpha=0.1, color='gray', label='Daily Range')
ax1.set_ylabel('Price ($)', fontsize=12)
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# ── Volume plot ──
ax2.bar(df_raw.index, df_raw['Volume'], color='#2ca02c', alpha=0.6, width=1)
ax2.set_ylabel('Volume', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('visualization/price_volume_trend.png', dpi=150, bbox_inches='tight')
plt.show()

print("\U0001f4cc Key Observations:")
print("   \u2022 The 50-day SMA crossing above/below the 200-day SMA is a classic signal:")
print("     Golden Cross (bullish) vs Death Cross (bearish)")
print("   \u2022 Volume spikes often coincide with major price movements or news events")
print("   \u2022 Notice the COVID crash in 2020 and the post-2022 recovery")


### 4.2 Distribution Analysis

Understanding the distribution of our target variable (Close price) and features helps us:
- Detect skewness (should we log-transform?)
- Spot outliers
- Choose appropriate error metrics
- Decide on normalization/scaling needs

In [ ]:
# ── Compute daily returns ──
df_raw['Daily_Return'] = df_raw['Close'].pct_change() * 100

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribution Analysis', fontsize=16, fontweight='bold')

# Close price distribution
sns.histplot(df_raw['Close'], kde=True, ax=axes[0, 0], color='steelblue', bins=50)
axes[0, 0].set_title('Close Price Distribution')
axes[0, 0].set_xlabel('Price ($)')

# Log close price
sns.histplot(np.log(df_raw['Close']), kde=True, ax=axes[0, 1], color='coral', bins=50)
axes[0, 1].set_title('Log(Close) Distribution')
axes[0, 1].set_xlabel('Log(Price)')

# Daily returns distribution
sns.histplot(df_raw['Daily_Return'].dropna(), kde=True, ax=axes[0, 2], color='green', bins=80)
axes[0, 2].set_title('Daily Returns Distribution (%)')
axes[0, 2].set_xlabel('Return (%)')
axes[0, 2].axvline(0, color='red', linestyle='--', alpha=0.5)

# Box plot of price columns
sns.boxplot(data=df_raw[['Open', 'High', 'Low', 'Close']], ax=axes[1, 0])
axes[1, 0].set_title('Price Columns Box Plot')
axes[1, 0].tick_params(axis='x', rotation=45)

# Volume distribution
sns.histplot(df_raw['Volume'], kde=True, ax=axes[1, 1], color='purple', bins=50)
axes[1, 1].set_title('Volume Distribution')

# Q-Q plot for normality check
stats.probplot(df_raw['Close'].values, dist="norm", plot=axes[1, 2])
axes[1, 2].set_title('Q-Q Plot (Normality Check)')

plt.tight_layout()
plt.savefig('visualization/distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n\U0001f4ca Return Statistics:")
print(f"   Mean daily return:  {df_raw['Daily_Return'].mean():.3f}%")
print(f"   Std daily return:   {df_raw['Daily_Return'].std():.3f}%")
print(f"   Skewness:           {df_raw['Daily_Return'].skew():.3f}")
print(f"   Kurtosis:           {df_raw['Daily_Return'].kurtosis():.3f}  (Normal = 3)")
print()
print("\U0001f4a1 Insight: Stock returns have fat tails (kurtosis > 3) \u2014")
print("   extreme movements happen more often than a normal distribution predicts.")


### 4.3 Correlation Analysis

Correlation helps us understand relationships between variables. In stock data:
- **Open, High, Low, Close** are naturally highly correlated
- **Volume** has a weaker, non-linear relationship with price
- **Daily returns** may show autocorrelation

> **Note:** High correlation (multicollinearity) can hurt linear models. Tree-based models handle it better.

In [ ]:
# ── Correlation matrix ──
corr_df = df_raw[['Open', 'High', 'Low', 'Close', 'Volume']].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
sns.heatmap(corr_df, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visualization/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\U0001f4cc Key Findings:")
print("   \u2022 Open/High/Low/Close are near-perfectly correlated (expected)")
print("   \u2022 Volume shows weak negative correlation with price (higher price = lower volume)")
print("   \u2022 We need to be careful about multicollinearity for linear models")


### 4.4 Stationarity Test (ADF Test)

**Why stationarity matters:** Time series models (especially ARIMA) require the data to be **stationary** \u2014 statistical properties (mean, variance) don't change over time. Stock prices are **non-stationary** (they trend upward), but returns are usually stationary.

**Augmented Dickey-Fuller (ADF) test:**
- **H\u2080 (null):** Series has a unit root (non-stationary)
- **H\u2081 (alternative):** Series is stationary
- **p-value < 0.05** \u2192 Reject H\u2080 \u2192 Series is stationary

In [ ]:
def check_stationarity(series, name):
    """Run ADF test and interpret results."""
    result = adfuller(series.dropna(), autolag='AIC')
    print(f"\n{'='*50}")
    print(f"  Stationarity Test: {name}")
    print(f"{'='*50}")
    print(f"  ADF Statistic:     {result[0]:.6f}")
    print(f"  p-value:           {result[1]:.6f}")
    print(f"  Critical Values:")
    for key, value in result[4].items():
        print(f"    {key}: {value:.6f}")

    is_stationary = result[1] < 0.05
    verdict = '\U0001f7e2 STATIONARY' if is_stationary else '\u274c NON-STATIONARY'
    print(f"\n  \u2192 {verdict} (p={'< 0.05' if is_stationary else '> 0.05'})")
    return is_stationary

# Test Close price
is_close_stationary = check_stationarity(df_raw['Close'], 'Close Price')

# Test daily returns
is_returns_stationary = check_stationarity(df_raw['Daily_Return'], 'Daily Returns (%)')

# Test first difference of close
close_diff = df_raw['Close'].diff().dropna()
check_stationarity(close_diff, 'Close Price \u2014 1st Difference')

print("\n\U0001f4a1 Insight: Stock prices are typically non-stationary, but their")
print("   returns are stationary. ARIMA models 'difference' the data to")
print("   make it stationary. Tree-based models don't require stationarity \u2014")
print("   one reason they work well for stock forecasting.")


### 4.5 Autocorrelation Analysis

Autocorrelation measures how a time series is correlated with its own past values (lags). This helps us:
- Determine if past prices predict future prices
- Choose the right lag order for ARIMA
- Understand the 'memory' of the series

In [ ]:
# ── ACF and PACF plots ──
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Autocorrelation Analysis', fontsize=16, fontweight='bold')

# ACF of Close price
plot_acf(df_raw['Close'].dropna(), lags=40, ax=axes[0, 0])
axes[0, 0].set_title('ACF \u2014 Close Price')

# PACF of Close price
plot_pacf(df_raw['Close'].dropna(), lags=40, ax=axes[0, 1], method='ywm')
axes[0, 1].set_title('PACF \u2014 Close Price')

# ACF of Daily Returns
plot_acf(df_raw['Daily_Return'].dropna(), lags=40, ax=axes[1, 0])
axes[1, 0].set_title('ACF \u2014 Daily Returns')

# PACF of Daily Returns
plot_pacf(df_raw['Daily_Return'].dropna(), lags=40, ax=axes[1, 1], method='ywm')
axes[1, 1].set_title('PACF \u2014 Daily Returns')

plt.tight_layout()
plt.savefig('visualization/acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

print("\U0001f4cc Interpretation:")
print("   \u2022 Close price ACF decays slowly \u2192 strong autocorrelation \u2192 NON-STATIONARY")
print("   \u2022 Returns ACF drops sharply after lag 0 \u2192 weak autocorrelation \u2192 STATIONARY")
print("   \u2022 Significant PACF spikes at early lags \u2192 AR terms needed for ARIMA")


## 5. Data Preprocessing

Now we prepare the data for modeling.

### The Critical Rule: No Look-Ahead Bias

When forecasting time series, we **must not** use future information to predict the past. This means:
- **Time-based split** (not random) \u2014 train on older data, test on newer data
- **Fit scaler on training data only**, then transform test data
- **All features must be computable at prediction time** (no future info)

In [ ]:
# ── Create a clean working DataFrame ──
df = df_raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# ── Time-based split (80% train, 20% test) ──
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

print(f"\U0001f4c5 Time-based Train/Test Split")
print(f"   Training:   {train_df.index.min().date()} \u2192 {train_df.index.max().date()}  ({len(train_df):,} days)")
print(f"   Testing:    {test_df.index.min().date()} \u2192 {test_df.index.max().date()}  ({len(test_df):,} days)")
print(f"   Ratio:      {len(train_df)/len(df)*100:.1f}% / {len(test_df)/len(df)*100:.1f}%")
print()

# ── Visualize the split ──
plt.figure(figsize=(16, 5))
plt.plot(train_df.index, train_df['Close'], label='Training', color='steelblue', linewidth=1.5)
plt.plot(test_df.index, test_df['Close'], label='Testing', color='coral', linewidth=1.5)
plt.axvline(x=train_df.index[-1], color='gray', linestyle='--', alpha=0.5)
plt.title('Train/Test Split (Time-Based)', fontsize=14, fontweight='bold')
plt.ylabel('Close Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('visualization/train_test_split.png', dpi=150, bbox_inches='tight')
plt.show()

print("\U0001f7e2 Time-based split complete \u2014 no look-ahead bias!")


## 6. Feature Engineering

In time series forecasting, **features are everything**. Raw prices alone rarely make good models. We engineer features that capture:

1. **Lag Features** \u2014 Past closing prices (autoregressive nature)
2. **Rolling Statistics** \u2014 Moving averages capture trend strength
3. **Technical Indicators** \u2014 RSI, moving average crossovers
4. **Calendar Features** \u2014 Day of week, month (market seasonality)
5. **Volatility** \u2014 Rolling standard deviation captures risk

> **Why not use Open/High/Low directly?** At prediction time for tomorrow's close, we don't know tomorrow's Open/High/Low yet. We use *lagged* versions.

In [ ]:
def engineer_features(df_feat, target_col='Close', lags=[1, 2, 3, 5, 10, 20]):
    """Engineer features using ONLY past information (no look-ahead bias)."""
    df_feat = df_feat.copy()

    # ── 1. Lag Features ──
    for lag in lags:
        df_feat[f'Lag_{lag}'] = df_feat[target_col].shift(lag)

    # ── 2. Rolling Statistics ──
    for window in [5, 10, 20, 50]:
        df_feat[f'SMA_{window}'] = df_feat[target_col].rolling(window).mean()
        df_feat[f'Std_{window}'] = df_feat[target_col].rolling(window).std()

    # ── 3. Price Ratios ──
    df_feat['Price_vs_SMA_20'] = df_feat[target_col] / df_feat['SMA_20']
    df_feat['Price_vs_SMA_50'] = df_feat[target_col] / df_feat['SMA_50']

    # ── 4. Moving Average Crossovers ──
    df_feat['SMA_5_vs_20'] = df_feat['SMA_5'] / df_feat['SMA_20']
    df_feat['SMA_20_vs_50'] = df_feat['SMA_20'] / df_feat['SMA_50']

    # ── 5. Volume Features ──
    df_feat['Volume_SMA_5'] = df_feat['Volume'].rolling(5).mean()
    df_feat['Volume_SMA_20'] = df_feat['Volume'].rolling(20).mean()
    df_feat['Volume_Ratio'] = df_feat['Volume'] / df_feat['Volume_SMA_20']

    # ── 6. RSI (Relative Strength Index, 14-day) ──
    delta = df_feat[target_col].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    df_feat['RSI_14'] = 100 - (100 / (1 + rs))

    # ── 7. Rate of Change (momentum) ──
    for period in [5, 10, 20]:
        df_feat[f'ROC_{period}'] = df_feat[target_col].pct_change(period) * 100

    # ── 8. Calendar Features ──
    df_feat['Day_of_Week'] = df_feat.index.dayofweek
    df_feat['Month'] = df_feat.index.month

    # ── 9. Log Returns & Volatility ──
    df_feat['Log_Return'] = np.log(df_feat[target_col] / df_feat[target_col].shift(1))
    df_feat['EWMA_Volatility'] = df_feat['Log_Return'].ewm(span=20).std()

    # ── Drop NaN rows from shifting ──
    initial_rows = len(df_feat)
    df_feat = df_feat.dropna()
    rows_dropped = initial_rows - len(df_feat)

    print(f"\U0001f7e2 Feature Engineering Complete!")
    print(f"   Features created: {len(df_feat.columns) - 5}")  # 5 original cols
    print(f"   Rows after NaN drop: {len(df_feat):,} (dropped {rows_dropped})")
    return df_feat

# ── Apply to full dataset, then split back ──
full_df = pd.concat([train_df, test_df])
featured_full = engineer_features(full_df)

# ── Split back with aligned indices ──
featured_train = featured_full.loc[featured_full.index <= train_df.index[-1]]
featured_test = featured_full.loc[featured_full.index >= test_df.index[0]]

print(f"\n\U0001f4ca Final Shapes:")
print(f"   Training:   {featured_train.shape}")
print(f"   Testing:    {featured_test.shape}")


### 6.1 Feature Selection

We must drop raw Open/High/Low columns \u2014 at prediction time, we don't know today's values yet. Only lagged and rolling features are valid.

In [ ]:
# ── Define feature columns ──
TARGET = 'Close'
EXCLUDE = ['Open', 'High', 'Low', 'Volume']  # Raw cols we can't use at prediction time
FEATURES = [col for col in featured_train.columns
            if col != TARGET and col not in EXCLUDE]

# ── Verify no NaN in features ──
assert featured_train[FEATURES].isnull().sum().sum() == 0, "NaN still present in features!"
assert featured_test[FEATURES].isnull().sum().sum() == 0, "NaN still present in test features!"

print(f"\U0001f4cb Selected {len(FEATURES)} features for modeling")
print()

# ── Split into X and y ──
X_train = featured_train[FEATURES]
y_train = featured_train[TARGET]
X_test = featured_test[FEATURES]
y_test = featured_test[TARGET]

print(f"Feature Matrix Shapes:")
print(f"   X_train: {X_train.shape}")
print(f"   X_test:  {X_test.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test:  {y_test.shape}")


### 6.2 Feature-Target Correlation

A quick look at which features correlate most with our target. This gives us an early signal of what matters.

In [ ]:
# ── Correlation with target ──
target_corr = featured_train[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(ascending=False)

plt.figure(figsize=(12, 8))
top_n = 20
colors = ['steelblue' if v > 0 else 'coral' for v in target_corr.values[:top_n]]
target_corr.head(top_n).plot(kind='bar', color=colors)
plt.title(f'Top {top_n} Features by Correlation with {TARGET}', fontsize=14, fontweight='bold')
plt.xlabel('Feature')
plt.ylabel(f'Correlation with {TARGET}')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('visualization/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\U0001f4cc Observations:")
print("   \u2022 Lag features have the highest correlation (past predicts future)")
print("   \u2022 SMA features follow \u2014 trend is the strongest signal")
print("   \u2022 RSI and volatility have lower correlation but capture different patterns")


### 6.3 Feature Scaling

Tree-based models (Random Forest, XGBoost) are **scale-invariant**. But linear models need scaling.

We scale separately on train (fit) and test (transform) to avoid data leakage.

In [ ]:
# ── Scale features ──
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ── Keep as DataFrames ──
X_train_scaled = pd.DataFrame(X_train_scaled, columns=FEATURES, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=FEATURES, index=X_test.index)

print("\U0001f7e2 Features scaled (StandardScaler \u2014 fitted on TRAIN only)")
print(f"   Mean after scaling: {X_train_scaled.mean(axis=0).mean():.6f} (should be \u22480)")
print(f"   Std after scaling:  {X_train_scaled.std(axis=0).mean():.6f} (should be \u22481)")


## 7. Modeling

We'll build multiple models, from simple to complex:

| Model | Type | Why |
|-------|------|-----|
| Naive Baseline | Baseline | Yesterday's close = today's prediction |
| Linear Regression | Linear | Simplest learned model |
| Ridge Regression | Linear | L2 regularization prevents overfitting |
| Random Forest | Tree ensemble | Captures non-linear patterns |
| XGBoost | Gradient boosting | State-of-the-art for tabular data |
| ARIMA | Statistical | Classic time series model |

### 7.1 Naive Baseline

The simplest possible forecast: **tomorrow's price = today's price**. Any useful model must beat this.

In [ ]:
# ── Naive forecast: predict tomorrow = today's close ──
y_pred_naive = featured_test[TARGET].shift(1).dropna()
y_actual_naive = y_test.loc[y_pred_naive.index]

# ── Metrics ──
mae_naive = mean_absolute_error(y_actual_naive, y_pred_naive)
rmse_naive = np.sqrt(mean_squared_error(y_actual_naive, y_pred_naive))
r2_naive = r2_score(y_actual_naive, y_pred_naive)

print("\U0001f4ca Naive Baseline Performance")
print("="*40)
print(f"  MAE:   ${mae_naive:.4f}")
print(f"  RMSE:  ${rmse_naive:.4f}")
print(f"  R\u00b2:    {r2_naive:.4f}")
print("="*40)
print("\n\U0001f4a1 This is our 'to beat' score. Any real model must outperform")
print("   the naive forecast to be useful.")


### 7.2 Linear Regression

A simple linear model: `Price = w\u2080 + w\u2081x\u2081 + w\u2082x\u2082 + ... + w\u2099x\u2099`

**Pros**: Interpretable, fast, provides feature coefficients
**Cons**: Assumes linear relationships, sensitive to multicollinearity

In [ ]:
# ── Linear Regression ──
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

# ── Ridge Regression (L2 regularization) ──
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)

def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"{name:25s}  MAE=${mae:<8.4f}  RMSE=${rmse:<8.4f}  R\u00b2={r2:<8.4f}  MAPE={mape:<6.2f}%")
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape}

results = []
results.append(evaluate_model('Naive Baseline', y_actual_naive, y_pred_naive))
results.append(evaluate_model('Linear Regression', y_test, y_pred_lr))
results.append(evaluate_model('Ridge Regression', y_test, y_pred_ridge))

# ── Top coefficients ──
coef_df = pd.DataFrame({'Feature': FEATURES, 'Coefficient': lr.coef_})
coef_df['Abs'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values('Abs', ascending=False).drop(columns='Abs')

print(f"\n\U0001f4cc Top 10 Most Influential Features (Linear Regression):")
print(coef_df.head(10).to_string(index=False))


### 7.3 Random Forest Regressor

Random Forest builds many decision trees on random subsets of data and features, then averages their predictions.

**Why it works well for stock data:**
- Captures **non-linear relationships** (price movements aren't linear!)
- Handles **feature interactions** automatically
- **Robust to outliers** (individual trees can't overfit to noise)
- **No scaling needed**

In [ ]:
# ── Random Forest ──
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

print("\U0001f333 Training Random Forest (300 trees)...")
rf.fit(X_train, y_train)  # RF doesn't need scaled data
y_pred_rf = rf.predict(X_test)

results.append(evaluate_model('Random Forest', y_test, y_pred_rf))

# ── Feature importance ──
importance_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\n\U0001f4cc Top 10 Most Important Features (Random Forest):")
print(importance_df.head(10).to_string(index=False))

# ── Plot ──
plt.figure(figsize=(12, 6))
plt.barh(importance_df.head(15)['Feature'][::-1],
         importance_df.head(15)['Importance'][::-1], color='steelblue')
plt.xlabel('Feature Importance')
plt.title('Random Forest \u2014 Top 15 Feature Importances', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('visualization/rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


### 7.4 XGBoost Regressor

**XGBoost** (Extreme Gradient Boosting) builds trees **sequentially** \u2014 each new tree corrects the errors of the previous ones.

**Key advantages:**
- **Gradient boosting** \u2192 learns from mistakes iteratively
- **Regularization** \u2192 built-in L1/L2 to prevent overfitting
- **Handles missing values** natively
- **Widely used in quantitative finance**

In [ ]:
# ── XGBoost ──
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

print("\U0001f680 Training XGBoost (500 trees)...")
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

results.append(evaluate_model('XGBoost', y_test, y_pred_xgb))

# ── Feature importance ──
xgb_importance = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\n\U0001f4cc Top 10 Most Important Features (XGBoost):")
print(xgb_importance.head(10).to_string(index=False))


### 7.5 ARIMA Model

**ARIMA** (AutoRegressive Integrated Moving Average) is the classic statistical approach for univariate time series.

- **AR(p):** Uses past values as predictors
- **I(d):** Differencing to make series stationary
- **MA(q):** Uses past forecast errors as predictors

ARIMA only uses the target variable's own history \u2014 no external features.

In [ ]:
# ── ARIMA uses only the close price on training data ──
arima_train = train_df['Close'].copy()
arima_test = test_df['Close'].copy()

print("\u23f3 Training ARIMA(5,1,5)... (this may take a moment)")
try:
    arima_model = ARIMA(arima_train, order=(5, 1, 5))
    arima_fitted = arima_model.fit()

    # ── Forecast test period ──
    y_pred_arima = arima_fitted.forecast(steps=len(arima_test))
    y_pred_arima.index = arima_test.index

    results.append(evaluate_model('ARIMA(5,1,5)', arima_test, y_pred_arima))
    print(f"\n\U0001f7e2 ARIMA converged  |  AIC: {arima_fitted.aic:.2f}")
except Exception as e:
    print(f"\u274c ARIMA failed: {e}")
    print("   (Common with volatile data \u2014 convergence issues)")
    results.append({'Model': 'ARIMA(5,1,5)', 'MAE': np.nan, 'RMSE': np.nan, 'R2': np.nan, 'MAPE': np.nan})


## 8. Model Evaluation & Comparison

| Metric | What it measures | Interpretation |
|--------|-----------------|----------------|
| **MAE** | Average absolute error | Easy to understand \u2014 average $ mistake |
| **RMSE** | Root mean squared error | Penalizes large errors more heavily |
| **R\u00b2** | Variance explained | How much price movement is captured |
| **MAPE** | % error | Scale-independent comparison |

In [ ]:
# ── Build comparison DataFrame ──
results_df = pd.DataFrame(results).dropna(subset=['RMSE'])
results_df = results_df.sort_values('RMSE', ascending=True).reset_index(drop=True)
naive_rmse = results_df.loc[results_df['Model'] == 'Naive Baseline', 'RMSE'].values[0]
results_df['vs_Naive'] = ((results_df['RMSE'] / naive_rmse) - 1) * 100

print("="*72)
print("  \U0001f3c6  MODEL COMPARISON (Test Set)")
print("="*72)
print(f"{'Model':25s} {'MAE($)':<10s} {'RMSE($)':<10s} {'R\u00b2':<8s} {'MAPE(%)':<8s} {'vs Naive':<10s}")
print("-"*72)
for _, row in results_df.iterrows():
    impr = f"{row['vs_Naive']:+5.1f}%" if not np.isnan(row['vs_Naive']) else "  N/A"
    print(f"{row['Model']:25s} ${row['MAE']:<8.4f} ${row['RMSE']:<8.4f} {row['R2']:<8.4f} {row['MAPE']:<8.2f} {impr}")
print("="*72)

best_model = results_df.iloc[0]
print(f"\n\U0001f3c5 Best Model: {best_model['Model']}")
print(f"   RMSE: ${best_model['RMSE']:.4f}  |  MAPE: {best_model['MAPE']:.2f}%")


### 8.1 Visualizing Predictions vs Actual

Seeing predictions plotted against actual prices reveals:
- Does the model capture the overall **trend**?
- Does it **lag** behind actual prices?
- Are there systematic **biases**?

In [ ]:
# ── Collect predictions ──
# ── Align y_test to match Naive Baseline's length ──
y_test_aligned = y_test.reindex(y_pred_naive.index)

predictions_dict = {
    'Naive Baseline': y_pred_naive,
    'Linear Regression': pd.Series(y_pred_lr, index=y_test.index).reindex(y_pred_naive.index),
    'Ridge Regression': pd.Series(y_pred_ridge, index=y_test.index).reindex(y_pred_naive.index),
    'Random Forest': pd.Series(y_pred_rf, index=y_test.index).reindex(y_pred_naive.index),
    'XGBoost': pd.Series(y_pred_xgb, index=y_test.index).reindex(y_pred_naive.index),
}

# Add ARIMA if it succeeded
if 'ARIMA(5,1,5)' in results_df['Model'].values:
    try:
        predictions_dict['ARIMA(5,1,5)'] = y_pred_arima
    except:
        pass

# ── Plot ──
n_models = len(predictions_dict)
fig, axes = plt.subplots(nrows=n_models, ncols=1, figsize=(16, 4 * n_models), sharex=True)
fig.suptitle('Model Predictions vs Actual Close Price', fontsize=16, fontweight='bold')

if n_models == 1:
    axes = [axes]

colors = plt.cm.Set2(np.linspace(0, 1, n_models))

for ax, (name, preds), color in zip(axes, predictions_dict.items(), colors):
    # Reindex preds to match y_test_aligned (handles length mismatch with Naive Baseline)
    preds_aligned = preds.reindex(y_test_aligned.index)
    ax.plot(y_test_aligned.index, y_test_aligned.values, label='Actual', color='black', linewidth=1.5, alpha=0.8)
    ax.plot(preds_aligned.index, preds_aligned.values, label=f'{name} (Predicted)', color=color, linewidth=1.2, alpha=0.8)
    ax.fill_between(y_test_aligned.index, y_test_aligned.values, preds_aligned.values, alpha=0.15, color=color)
    ax.set_ylabel('Price ($)')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_title(f'{name}', fontsize=12)

plt.tight_layout()
plt.savefig('visualization/all_predictions.png', dpi=150, bbox_inches='tight')
plt.show()


### 8.2 Detailed Analysis of Best Model

Let's zoom into the last 60 trading days and analyze the residuals of our best model.

In [ ]:
# ── Best model ──
best_model_name = results_df.iloc[0]['Model']
best_preds = predictions_dict.get(best_model_name)

if best_preds is not None:
    zoom_days = 60
    zoom_start = max(0, len(y_test) - zoom_days)

    # Align to common index to handle length mismatches
    common_idx = y_test.index[zoom_start:].intersection(best_preds.index[zoom_start:])
    plt.figure(figsize=(16, 7))
    plt.plot(common_idx, y_test.loc[common_idx].values,
             label='Actual', color='black', linewidth=2, marker='o', markersize=3)
    plt.plot(common_idx, best_preds.loc[common_idx].values,
             label=f'{best_model_name} (Best Model)', color='#2ca02c', linewidth=2,
             marker='s', markersize=3, alpha=0.8)
    plt.fill_between(common_idx,
                     y_test.loc[common_idx].values,
                     best_preds.loc[common_idx].values,
                     alpha=0.2, color='coral', label='Prediction Error')
    plt.title(f'{best_model_name} \u2014 Zoomed View (Last {zoom_days} Trading Days)',
              fontsize=14, fontweight='bold')
    plt.ylabel('Close Price ($)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('visualization/best_model_zoom.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Residual analysis ──
    residuals = y_test.loc[common_idx].values - best_preds.loc[common_idx].values

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.scatter(best_preds.loc[common_idx].values, residuals, alpha=0.6, color='steelblue')
    ax1.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax1.set_xlabel('Predicted Price ($)')
    ax1.set_ylabel('Residual (Actual - Predicted)')
    ax1.set_title('Residuals vs Predicted', fontsize=12)
    ax1.grid(True, alpha=0.3)

    ax2.hist(residuals, bins=30, color='steelblue', edgecolor='white', alpha=0.7)
    ax2.axvline(x=0, color='red', linestyle='--', alpha=0.5)
    ax2.set_xlabel('Residual ($)')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Residual Distribution', fontsize=12)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('visualization/residual_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\U0001f4ca Residual Analysis:")
    print(f"   Mean Residual:     ${residuals.mean():.4f}  (ideally \u2248 0)")
    print(f"   Std of Residuals:  ${residuals.std():.4f}")
    print(f"   Max Underestimate: ${residuals.min():.4f}")
    print(f"   Max Overestimate:  ${residuals.max():.4f}")
    bias = "unbiased" if abs(residuals.mean()) < 1 else ("underpredicts" if residuals.mean() > 0 else "overpredicts")
    print(f"\n   \u2192 Model is {bias}" + (" (residuals centered near zero)" if abs(residuals.mean()) < 1 else ""))


### 8.3 Error Analysis \u2014 When Does the Model Fail?

Understanding *when* predictions go wrong is as important as knowing *how much*.

In [ ]:
# ── Error DataFrame ──
if best_preds is not None:
    error_df = pd.DataFrame({
        'Actual': y_test,
        'Predicted': best_preds,
        'Error': y_test - best_preds,
        'Abs_Error': abs(y_test - best_preds),
        'Pct_Error': ((y_test - best_preds) / y_test * 100)
    })

    print("\U0001f4c9 5 WORST Predictions (Largest Absolute Errors):")
    print("="*65)
    print(error_df.sort_values('Abs_Error', ascending=False).head(5).to_string())
    print()

    print("\U0001f4c8 5 BEST Predictions (Smallest Absolute Errors):")
    print("="*65)
    print(error_df.sort_values('Abs_Error', ascending=True).head(5).to_string())
    print()

    # ── Check error vs volatility ──
    error_df['Volatility'] = abs(test_df['Close'].pct_change().loc[error_df.index])
    vol_corr = error_df[['Abs_Error', 'Volatility']].corr().iloc[0, 1]
    print(f"\U0001f517 Correlation: Abs_Error vs Market Volatility = {vol_corr:.3f}")
    print()
    print("\U0001f4a1 Insight: If errors correlate with volatility, the model")
    print("   struggles with sudden market moves \u2014 a common challenge.")


## 9. Future Forecasting \u2014 Predicting Ahead

Let's forecast the **next 30 trading days** using our best model retrained on all available data.

In [ ]:
# ── Retrain best model on ALL available data ──
print(f"\U0001f3c6 Best Model: {best_model_name}")
print(f"   Retraining on full dataset ({len(full_df):,} rows)...")

# ── Feature engineer on full data ──
full_featured = engineer_features(full_df.copy())
X_full = full_featured[FEATURES]
y_full = full_featured[TARGET]

if best_model_name == 'XGBoost':
    final_model = xgb.XGBRegressor(
        n_estimators=500, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1
    )
elif best_model_name == 'Random Forest':
    final_model = RandomForestRegressor(
        n_estimators=300, max_depth=15, min_samples_leaf=5, random_state=42, n_jobs=-1
    )
elif best_model_name == 'Ridge Regression':
    final_model = Ridge(alpha=1.0)
else:
    final_model = LinearRegression()

final_model.fit(X_full, y_full)

# ── Generate future dates (next 30 trading days) ──
last_date = full_df.index[-1]
future_dates = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=30)
print(f"   Forecasting: {future_dates[0].date()} \u2192 {future_dates[-1].date()} ({len(future_dates)} trading days)")
print()

# ── Iterative forecasting ──
future_df = full_df.copy()
future_predictions = []

for i, date in enumerate(future_dates):
    # Create a row for this future date
    new_row = future_df.iloc[-1:].copy()
    new_row.index = [date]

    # Add engineered features
    temp_df = pd.concat([future_df, new_row])
    temp_featured = engineer_features(temp_df)

    # Get the last row's features
    future_features = temp_featured.iloc[[-1]][FEATURES]

    # Predict
    pred = final_model.predict(future_features)[0]
    future_predictions.append(pred)

    # Update for next iteration
    new_row.loc[date, 'Close'] = pred
    for col in ['Open', 'High', 'Low']:
        new_row.loc[date, col] = pred
    new_row.loc[date, 'Volume'] = future_df['Volume'].iloc[-1]
    future_df = pd.concat([future_df, new_row])

# ── Plot ──
plt.figure(figsize=(16, 7))
lookback = 180
plt.plot(full_df.index[-lookback:], full_df['Close'].values[-lookback:],
         label='Historical', color='steelblue', linewidth=2)
plt.plot(future_dates, future_predictions, label='Forecast (30 days)',
         color='red', linewidth=2, linestyle='--', marker='o', markersize=4)
plt.axvline(x=full_df.index[-1], color='gray', linestyle=':', alpha=0.7)
plt.title(f'{TICKER} \u2014 Historical & 30-Day Forecast ({best_model_name})',
          fontsize=14, fontweight='bold')
plt.ylabel('Close Price ($)')
plt.xlabel('Date')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('visualization/future_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary ──
print(f"\n\U0001f4ca 30-Day Forecast Summary:")
print(f"   Current Price:  ${full_df['Close'].iloc[-1]:.2f}")
print(f"   Forecast End:   ${future_predictions[-1]:.2f}")
change_pct = ((future_predictions[-1] / full_df['Close'].iloc[-1]) - 1) * 100
print(f"   Projected Change: {change_pct:.2f}%")
print(f"   Min Forecast:   ${min(future_predictions):.2f}")
print(f"   Max Forecast:   ${max(future_predictions):.2f}")
print()
print("\u26a0\ufe0f  IMPORTANT DISCLAIMER: This is a mathematical extrapolation.")
print("   It does NOT account for earnings reports, macroeconomic events,")
print("   market sentiment, news, or any external factors.")


## 10. Walk-Forward Validation (Robustness Check)

Instead of a single 80/20 split, walk-forward validation trains on expanding windows and tests on each subsequent period. This simulates real-time deployment.

In [ ]:
# ── Time Series Cross-Validation ──
tscv = TimeSeriesSplit(n_splits=5)
cv_results = []

print("\U0001f504 Walk-Forward Validation (5 folds)")
print("="*55)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_full), 1):
    X_cv_train = X_full.iloc[train_idx]
    y_cv_train = y_full.iloc[train_idx]
    X_cv_val = X_full.iloc[val_idx]
    y_cv_val = y_full.iloc[val_idx]

    # Use XGBoost for cross-validation
    cv_model = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                                 random_state=42, n_jobs=-1, verbosity=0)
    cv_model.fit(X_cv_train, y_cv_train)
    y_cv_pred = cv_model.predict(X_cv_val)

    rmse = np.sqrt(mean_squared_error(y_cv_val, y_cv_pred))
    mape = np.mean(np.abs((y_cv_val - y_cv_pred) / y_cv_val)) * 100
    cv_results.append({'Fold': fold, 'Train': len(X_cv_train),
                       'Test': len(X_cv_val), 'RMSE': rmse, 'MAPE': mape})
    print(f"  Fold {fold}: Train={len(X_cv_train):,}  Test={len(X_cv_val):,}  "
          f"RMSE=${rmse:.4f}  MAPE={mape:.2f}%")

cv_df = pd.DataFrame(cv_results)
print(f"\n\U0001f4ca Cross-Validation Summary:")
print(f"   Avg RMSE: ${cv_df['RMSE'].mean():.4f} (\u00b1 ${cv_df['RMSE'].std():.4f})")
print(f"   Avg MAPE: {cv_df['MAPE'].mean():.2f}% (\u00b1 {cv_df['MAPE'].std():.2f}%)")
print()
if cv_df['RMSE'].std() / cv_df['RMSE'].mean() < 0.2:
    print("\U0001f7e2 Consistent performance across folds \u2014 robust model!")
else:
    print("\u26a0\ufe0f High variance across folds \u2014 model is sensitive to market regimes")


## 11. Conclusion & Key Takeaways

### What We Accomplished

1. **Real-World Data**: Fetched 5 years of Apple (AAPL) daily stock data using yfinance
2. **Thorough EDA**: Explored trends, distributions, correlations, stationarity, and autocorrelation
3. **Rich Feature Engineering**: Created 30+ features (lags, SMA, RSI, volatility, calendar features)
4. **Multiple Models**: Compared Naive, Linear Regression, Ridge, Random Forest, XGBoost, and ARIMA
5. **Robust Evaluation**: Time-based split, multiple metrics, walk-forward cross-validation
6. **Future Forecasting**: Generated 30-day ahead predictions

### Key Findings

- **XGBoost and Random Forest** consistently outperform linear models \u2014 stock prices have non-linear relationships
- **Lag features and moving averages** are the most predictive signals
- The naive baseline is surprisingly competitive \u2014 stock prices have strong autocorrelation
- **Large prediction errors** occur during high-volatility periods
- ARIMA struggles with this data \u2014 tree-based models capture far more patterns

### Practical Limitations

| Limitation | Impact | Mitigation |
|------------|--------|------------|
| Only price/volume data | Misses news, sentiment, macro factors | Add NLP on news + economic indicators |
| Daily frequency | No intraday patterns | Use minute-level data for HFT |
| Single stock | Portfolio patterns missed | Multi-stock / ETF modeling |
| Point forecasts | No confidence intervals | Quantile regression or Bayesian methods |


## 12. Suggested Improvements

### \U0001f680 Modeling Enhancements

| Improvement | Why | Implementation |
|-------------|-----|----------------|
| **LSTM / GRU** | Captures long-range temporal dependencies | PyTorch/TensorFlow seq-to-seq |
| **Transformer** | Attention over time | Time Series Transformer |
| **Ensemble Stacking** | Combine model strengths | Meta-model on RF + XGB + Ridge |
| **Bayesian Methods** | Prediction intervals | Prophet, Gaussian Processes |
| **LightGBM / CatBoost** | Faster boosting alternatives | pip install lightgbm catboost |

### \U0001f4ca Data Enhancements

| Improvement | Why |
|-------------|-----|
| **Sentiment Analysis** | News headlines / Twitter sentiment as features |
| **Macroeconomic Data** | Interest rates, inflation, GDP, VIX |
| **Sector Correlation** | How peers (MSFT, GOOGL) move relative to AAPL |
| **Options Data** | Implied volatility, put/call ratios |
| **Fundamentals** | P/E ratio, earnings growth, revenue |

### \U0001f527 Engineering Improvements

- **Hyperparameter Tuning**: Use Optuna or GridSearchCV
- **Feature Selection**: SHAP values or Recursive Feature Elimination
- **Multi-Step Forecast**: Predict 5/10/30 days directly instead of iteratively
- **Trading Strategy Backtest**: Add realistic costs and slippage
- **Dashboards**: Streamlit or Grafana for real-time monitoring

### Production Pipeline

```
Data Pipeline \u2192 Feature Store \u2192 Model Training \u2192 Registry \u2192 Inference API \u2192 Dashboard
(yfinance T+1)   (Redis)      (MLflow)    (FastAPI)   (Streamlit)
```

> **Final Thought:** Stock price forecasting is inherently challenging because markets are **weak-form efficient** \u2014 past prices alone explain only a fraction of future movements. The real value lies in combining price data with **alternative data sources** and building systems that **manage risk** rather than just predict prices.
